In [1]:
import _referAsMain
import paths_cfg
from dataset import svg_dataset
import numpy
from torch.utils.data import DataLoader

added '/autofs/unitytravail/travail/lcurtis/M2INFO/PFE/PFE_LLM_art_generation' to import paths


In [2]:
dataset = svg_dataset.SVGDataset(
    paths_cfg.OUR_DATASET_DIRECTORY,
    context_size=4096,
    use_gcode=True,
    use_relative_gcode=False,
)

Modification du SGVDataset, donc les deux tests suivants ne sont plus valide

In [3]:
# taille des differents svg
print(f"Nombre de samples : {len(dataset.samples)}")
a = sorted([(len(svg.txt), svg.svg_file) for svg in dataset.samples], reverse=True)
print(f"total chars: {sum(t[0] for t in a):_d}")
sizes = numpy.array([t[0] for t in a])
print(f"mean sizes: {int(sizes.mean()):_d}, std:{int(sizes.std()):_d}")
print(numpy.percentile(sizes, [10, 25, 50, 75, 90]))
print("\n".join(map(lambda t: f"(nb chars: {t[0]:_d}, path: {t[1]})", a)))

Nombre de samples : 100
total chars: 104_802_360
mean sizes: 1_048_023, std:1_186_719
[  18511.  235035.  616795. 1344950. 2966751.]
(nb chars: 4_991_095, path: /autofs/unitytravail/travail/lcurtis/M2INFO/PFE/PFE_LLM_art_generation/dataset/samples_100/0045_circle_packing.svg)
(nb chars: 4_730_975, path: /autofs/unitytravail/travail/lcurtis/M2INFO/PFE/PFE_LLM_art_generation/dataset/samples_100/0098_circle_packing.svg)
(nb chars: 4_175_095, path: /autofs/unitytravail/travail/lcurtis/M2INFO/PFE/PFE_LLM_art_generation/dataset/samples_100/0091_circle_packing.svg)
(nb chars: 3_868_555, path: /autofs/unitytravail/travail/lcurtis/M2INFO/PFE/PFE_LLM_art_generation/dataset/samples_100/0030_circle_packing.svg)
(nb chars: 3_647_855, path: /autofs/unitytravail/travail/lcurtis/M2INFO/PFE/PFE_LLM_art_generation/dataset/samples_100/0085_circle_packing.svg)
(nb chars: 3_507_135, path: /autofs/unitytravail/travail/lcurtis/M2INFO/PFE/PFE_LLM_art_generation/dataset/samples_100/0097_subdivision.svg)
(nb ch

In [4]:
55_582_055 / (4096 / 2)

27139.67529296875

In [ ]:
# solution pour constituer les samples pour entrainer sur le dataset
def tokenize(svg_text):
    return list(map(ord, svg_text))


context = 4096
# "<|EndOfText|>".join((dataset[i] for i in range(1)))
a = dataset.samples[0].txt
tokens = tokenize(a)
b = [tokens[(i * context // 2) : (i * context // 2) + context] for i in range(50)]
c = [a[(i * context // 2) : (i * context // 2) + context] for i in range(50)]
print("\n".join(map(lambda t: f"{t[0]}\n{t[1]}", zip(b, c))))

[60, 115, 118, 103, 32, 120, 109, 108, 110, 115, 58, 120, 108, 105, 110, 107, 61, 34, 104, 116, 116, 112, 58, 47, 47, 119, 119, 119, 46, 119, 51, 46, 111, 114, 103, 47, 49, 57, 57, 57, 47, 120, 108, 105, 110, 107, 34, 32, 120, 109, 108, 110, 115, 61, 34, 104, 116, 116, 112, 58, 47, 47, 119, 119, 119, 46, 119, 51, 46, 111, 114, 103, 47, 50, 48, 48, 48, 47, 115, 118, 103, 34, 32, 115, 116, 121, 108, 101, 61, 34, 102, 105, 108, 108, 45, 111, 112, 97, 99, 105, 116, 121, 58, 49, 59, 32, 99, 111, 108, 111, 114, 45, 114, 101, 110, 100, 101, 114, 105, 110, 103, 58, 97, 117, 116, 111, 59, 32, 99, 111, 108, 111, 114, 45, 105, 110, 116, 101, 114, 112, 111, 108, 97, 116, 105, 111, 110, 58, 97, 117, 116, 111, 59, 32, 116, 101, 120, 116, 45, 114, 101, 110, 100, 101, 114, 105, 110, 103, 58, 97, 117, 116, 111, 59, 32, 115, 116, 114, 111, 107, 101, 58, 98, 108, 97, 99, 107, 59, 32, 115, 116, 114, 111, 107, 101, 45, 108, 105, 110, 101, 99, 97, 112, 58, 115, 113, 117, 97, 114, 101, 59, 32, 115, 116, 114,

fichiers SVG(list[svgSample]) -> Tokenisation (1 list[int] par SVG) -> split

Test suite à l'utilisation du tokeniser (SVGDataset) :

In [6]:
CONTEXT_SIZE = 4096

ds = svg_dataset.SVGDataset(
    CURRENT_DIRECTORY.joinpath("dataset/samples"), context_size=CONTEXT_SIZE
)  # using default tokenization

print(f"context_size         : {CONTEXT_SIZE}")
print(f"Nombre de chunks     : {len(ds)}")

context_size         : 4096
Nombre de chunks     : 4232


In [7]:
idx = 4231
sample = ds.chunks[idx]

print(f"Longueur du chunk [{idx}] : {len(sample.tokens)} tokens")
print(f"Premiers tokens          : {sample.tokens[:20]}")
print(f"Texte correspondant      : {sample.text[:200]}")

Longueur du chunk [4231] : 4096 tokens
Premiers tokens          : [ 59  34  32 120  49  61  34  51  54  51  46  49  57  48  54  34  32 120
  50  61]
Texte correspondant      : ;" x1="363.1906" x2="363.1906" y1="373.4466"/><circle r="0" style="fill:none;" cx="545.9686" cy="397.713"/><circle r="2.227" style="fill:none;" cx="545.9687" cy="397.7129"/><circle r="4.4539" style="f


### Tests suite à l'implémentation du dataloader par batch

In [8]:
BATCH_SIZE = 4
dataloader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f"Batch size : {BATCH_SIZE}")
print(f"Nombre de batches : {len(dataloader)}")

Batch size : 4
Nombre de batches : 1058


In [12]:
batch = next(iter(dataloader))
print(f"Batch shape : {batch['tokens'].shape}")
print(f"Batch dtype : {batch['tokens'].dtype}")
print(f"elts index  : {batch['datasetIndex']}")

Batch shape : torch.Size([4, 4096])
Batch dtype : torch.int32
elts index  : tensor([ 862, 4165,  992, 4088])


In [13]:
for i in range(BATCH_SIZE):
    print(f"\n--- Sample {i} ---")
    print(f"Tokens : {batch['tokens'][i][:20]}")
    print(f"Texte : {ds.chunks[batch['datasetIndex'][i]].text}")


--- Sample 0 ---
Tokens : tensor([ 99, 121,  61,  34,  52,  54,  49,  46,  50,  48,  52,  54,  34,  47,
         62,  60,  99, 105, 114,  99], dtype=torch.int32)
Texte : cy="461.2046"/><circle r="7.2886" style="fill:none;" cx="522.6965" cy="224.4499"/><line y2="231.3208" style="fill:none;" x1="525.1285" x2="520.2646" y1="217.579"/><circle r="2.3806" style="fill:none;" cx="457.8167" cy="266.9334"/><circle r="5.2535" style="fill:none;" cx="575.0727" cy="797.5969"/><circle r="3.1886" style="fill:none;" cx="597.7339" cy="164.6432"/><line y2="164.6543" style="fill:none;" x1="594.5453" x2="600.9225" y1="164.6321"/><circle r="17.9449" style="fill:none;" cx="181.383" cy="422.5598"/><circle r="11.4693" style="fill:none;" cx="181.383" cy="422.5598"/><circle r="4.9937" style="fill:none;" cx="181.383" cy="422.5599"/><circle r="15.5203" style="fill:none;" cx="739.5697" cy="696.0315"/><circle r="1.164" style="fill:none;" cx="739.5697" cy="696.0315"/><circle r="4.7001" style="fill:none;" cx="775.749